## Delivery Delay Hypothesis

In [ ]:
# Import required Libraries
import numpy as np
import pandas as pd

In [ ]:
# Reading olist_orders_dataset.csv
orders_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv", parse_dates=orders_date_cols)

In [ ]:
# Reading olist_order_reviews_dataset.csv
reviews_date_cols = ["review_creation_date", "review_answer_timestamp"]
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv", parse_dates=reviews_date_cols)

In [ ]:
# Filtering only delivered orders
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

In [ ]:
# Dropping the rows where estimated_delivery_date was missing
orders_delivered = orders_delivered.dropna(subset=['order_delivered_customer_date'])

In [ ]:
# Creating new column for delay days
orders_delivered["delivery_delay_days"] =  (orders_delivered["order_delivered_customer_date"]-
                                            orders_delivered["order_estimated_delivery_date"]).dt.days

In [ ]:
# Creating new column delay_status
order_date_bins = [-np.inf, -1, 0, 3, 7, np.inf]

order_delay_labels = ["Early", "On time", "1-3 Days Late", "4-7 Days Late", "7+ Days Late"]

orders_delivered["delay_status"] = pd.cut(orders_delivered["delivery_delay_days"],
       order_date_bins, 
       labels=order_delay_labels,
       right=True)

In [ ]:
# Merging orders_delivered and reviews dataframe
orders_reviews = pd.merge(orders_delivered, reviews, on="order_id", how="left")

In [ ]:
# Calculating mean of review_score for delay_status
orders_reviews.groupby("delay_status")["review_score"].mean().round(3)

#### Observation:
Early orders get an average of `4.294` stars and highest in the list. On time delivered orders get an average of `4.033` stars. But there are less average of reviews score for orders delivered late.

In [ ]:
# Export processed data
# orders_reviews.to_csv("../data/processed/delivery_delay_hypothesis.csv", index=False)